# Lab 03: Data Acquisition and Initial Data Inspection

**Course:** BDS2203 — Data Processing  
**Guided demonstration:** Bangladesh population data from the World Bank API  
**Independent exercise:** Daily weather data for Dhaka from the Open-Meteo API

## Learning objectives
By the end of this lab, you will be able to:
- Send an HTTP GET request to a public API and check the response.
- Parse a JSON response and identify the relevant data.
- Convert API data into a Pandas DataFrame.
- Inspect rows, columns, data types, and missing values.
- Save a DataFrame as a CSV file.

**Instructions:** Run the cells in order. Complete the exercises after the demonstrations. Use the hints and references, but write your own code for the independent exercise.


## Part A — Instructor-guided demonstration: Bangladesh population

### 1. Import the libraries

We will use `requests` to access an API and `pandas` to organise and inspect tabular data. These libraries are available by default in Google Colab.


In [1]:
%pip install requests pandas

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.8 MB 9.3 MB/s eta 0:00:01
   ------------- -------------------------- 3.4/9.8 MB 8.7 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.8 MB 9.6 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.8 MB 9.2 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.8 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 8.4 MB/s  0:00:01
   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   ---- ----------------------------------- 1.6/12.7 MB 8.9 MB/s eta 0:00:02
   ---------- ----------------------------- 3.4/12.7 MB 8.6 MB/s eta 0:00:02
   ---------------- ----------------------- 5.2/12.7 MB 8.8 MB/s eta 0:00:01
   ---------------------- ----------------- 7.1/12.7 MB 8.7 MB/s eta 0:00:01
   ------------------------- -------------- 8.1/12.7 MB 8.4 MB/s eta 0:00:01
   --------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import requests
import pandas as pd


### 2. Request data from the World Bank API

An API allows a program to request data from an online service.

- `BGD` is the World Bank country code for Bangladesh.
- `SP.POP.TOTL` is the total population indicator.
- `format=json` requests a JSON response.
- `per_page=100` requests up to 100 records on this page.

**Dataset page:** [World Bank — Bangladesh Population](https://data.worldbank.org/indicator/SP.POP.TOTL?locations=BD)  
**API endpoint:** [World Bank API](https://api.worldbank.org/v2/country/BGD/indicator/SP.POP.TOTL?format=json&per_page=100)


In [4]:
url = (
    "https://api.worldbank.org/v2/country/BGD/"
    "indicator/SP.POP.TOTL?format=json&per_page=100"
)

response = requests.get(url, timeout=20)

print("HTTP status code:", response.status_code)
response.raise_for_status()


HTTP status code: 200


### 3. Parse the JSON response

The response contains two main elements:
- `metadata` describes the response, such as the total number of records.
- `records` contains the actual population observations.

Each population record includes fields such as `date` and `value`.


In [5]:
data = response.json()

metadata = data[0]
records = data[1]

print("Total records reported by API:", metadata["total"])
print("Records received:", len(records))
print("\nExample record:")
print(records[0])


Total records reported by API: 66
Records received: 66

Example record:
{'indicator': {'id': 'SP.POP.TOTL', 'value': 'Population, total'}, 'country': {'id': 'BD', 'value': 'Bangladesh'}, 'countryiso3code': 'BGD', 'date': '2025', 'value': 175686899, 'unit': '', 'obs_status': '', 'decimal': 0}


### Checkpoint 1 — Understand the API response

Use the output from the previous cell to answer the questions below. Write your answers as comments; no additional Python code is required.

1. Which field represents the observation year?
2. Which field contains the population value?
3. How many records were received in this response?


In [ ]:
# 1. Field representing the year:

# 2. Field containing the population value:

# 3. Number of records received:


### 4. Convert the records into a DataFrame

A DataFrame organises data into rows and columns. The original API records contain several fields, but for this demonstration we will keep only the year and population value and give them clearer column names.


In [6]:
df_population = pd.DataFrame(records)

df_population = df_population[["date", "value"]].rename(
    columns={
        "date": "year",
        "value": "population"
    }
)

display(df_population.head())


,year,population
0,2025,175686899
1,2024,173562364
2,2023,171466990
3,2022,169384897
4,2021,167658854


### 5. Inspect the dataset

These commands are useful for an initial inspection:
- `head()` displays the first rows.
- `shape` returns the number of rows and columns.
- `columns` lists the column names.
- `dtypes` shows the data types.
- `isnull().sum()` counts missing values in each column.


In [7]:
print("First five records:")
display(df_population.head())

print("Dataset dimensions:", df_population.shape)

print("\nColumn names:")
print(df_population.columns.tolist())

print("\nData types:")
print(df_population.dtypes)

print("\nMissing values in each column:")
print(df_population.isnull().sum())


First five records:


,year,population
0,2025,175686899
1,2024,173562364
2,2023,171466990
3,2022,169384897
4,2021,167658854


Dataset dimensions: (66, 2)

Column names:
['year', 'population']

Data types:
year            str
population    int64
dtype: object

Missing values in each column:
year          0
population    0
dtype: int64


### 6. Save the DataFrame as a CSV file

CSV is a common format for storing tabular data. The `index=False` argument prevents Pandas from adding the DataFrame row index as an extra column.


In [9]:
df_population.to_csv("bangladesh_population.csv", index=False)
print("Dataset saved as bangladesh_population.csv")


Dataset saved as bangladesh_population.csv


## Part B — Student exercise: Acquire and inspect Dhaka weather data

In this exercise, apply the same workflow to a different API. Retrieve a seven-day weather forecast for Dhaka using Open-Meteo, convert the daily weather information into a DataFrame, inspect it, and save it as a CSV file.

**API documentation:** [Open-Meteo Forecast API](https://open-meteo.com/en/docs)  
**API endpoint:** [Seven-day Dhaka forecast](https://api.open-meteo.com/v1/forecast?latitude=23.8103&longitude=90.4125&daily=temperature_2m_max,temperature_2m_min,precipitation_sum&timezone=Asia%2FDhaka&forecast_days=7)

The response contains a `daily` section with arrays for:
- `time` — date
- `temperature_2m_max` — maximum temperature in °C
- `temperature_2m_min` — minimum temperature in °C
- `precipitation_sum` — total precipitation in mm

### Exercise 1 — Send a request and inspect the response

Complete the code cell:
1. Send a GET request to the API endpoint.
2. Print the HTTP status code.
3. Check for an unsuccessful response.
4. Parse the response as JSON and display its top-level keys.

**Hints**
- Follow the World Bank request example above.
- Use `requests.get()`, `response.status_code`, `response.raise_for_status()`, and `response.json()`.
- The API endpoint is provided below.


In [10]:
weather_url = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=23.8103&longitude=90.4125"
    "&daily=temperature_2m_max,temperature_2m_min,precipitation_sum"
    "&timezone=Asia%2FDhaka&forecast_days=7"
)

# 1. Send the GET request
weather_response = requests.get(weather_url, timeout=20)

# 2. Print the HTTP status code
print("HTTPS STATUS CODE",weather_response.status_code)

# 3. Check whether the response was successful
weather_response.raise_for_status()
print("request successfull")

# 4. Parse the JSON response and display its top-level keys
weather_response_json = weather_response.json()
print(weather_response_json.keys())



HTTPS STATUS CODE 200
request successfull
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])


### Exercise 2 — Create a weather DataFrame

Extract the daily weather information from the JSON response and create a DataFrame with exactly these columns:

- `date`
- `temperature_max_c`
- `temperature_min_c`
- `precipitation_mm`

Display the first five records.

**Hints**
- The daily information is stored under the `daily` key in the JSON response.
- Use `pd.DataFrame()` to combine the daily arrays into rows.
- The values for the requested fields are listed in the API description above.


In [11]:
# Extract the daily weather information
daily=weather_response_json["daily"]

# Create a DataFrame with the four required columns
df_weather = pd.DataFrame({
    "date": daily["time"],
    "temperature_max_c": daily["temperature_2m_max"],
    "temperature_min_c": daily["temperature_2m_min"],
    "precipitation_mm": daily["precipitation_sum"]
})


# Display the first five records
print(df_weather.head())


         date  temperature_max_c  temperature_min_c  precipitation_mm
0  2026-10-16               34.9               26.5               0.1
1  2026-10-17               35.2               26.3               0.0
2  2026-10-18               34.8               26.3               0.0
3  2026-10-19               34.9               25.0               0.0
4  2026-10-20               34.0               23.7               0.0


### Exercise 3 — Inspect the weather dataset

Using your weather DataFrame, complete the following tasks:

1. Display the number of rows and columns.
2. Display the column names.
3. Display the data types.
4. Count missing values in each column.

**Hints**
- Use the same Pandas commands demonstrated in Part A: `shape`, `columns`, `dtypes`, and `isnull().sum()`.


In [12]:
# 1. Display the dataset dimensions
print("Dataset dimensions : ",df_weather.shape)

# 2. Display the column names
print("Coloumn Name",df_weather.columns.tolist())

# 3. Display the data types
print("Data type :",df_weather.dtypes)


# 4. Count missing values
print("MISSING VALUES :",df_weather.isnull().sum())


Dataset dimensions :  (7, 4)
Coloumn Name ['date', 'temperature_max_c', 'temperature_min_c', 'precipitation_mm']
Data type : date                     str
temperature_max_c    float64
temperature_min_c    float64
precipitation_mm     float64
dtype: object
MISSING VALUES : date                 0
temperature_max_c    0
temperature_min_c    0
precipitation_mm     0
dtype: int64


### Exercise 4 — Save the weather dataset

Save the weather DataFrame as **`dhaka_weather.csv`**.

1. Save the file using `to_csv()` and `index=False`.
2. Check whether the file exists.
3. In Google Colab, open the **Files** panel to download the CSV file.

**Hints**
- You may use Python's `os` module and `os.path.exists()` to check the file.
- The population CSV example in Part A shows how to save a DataFrame.


In [13]:
import os
# Save the weather DataFrame as "dhaka_weather.csv"
df_weather.to_csv("dhaka_weather.csv", index=False)
print("Dataset saved as dhaka_weather.csv")

# Check whether the file exists
if os.path.exists("dhaka_weather.csv"):
  print("File exists")
else:
  print("File does not exist")


Dataset saved as dhaka_weather.csv
File exists


## Submission checklist

Submit your completed notebook (`.ipynb`) with the code and relevant outputs visible.

Before submitting:
- Run the cells in order.
- Complete Checkpoint 1 and all four weather exercises.
- Ensure the API request succeeds and the notebook runs without errors.
- Save and download `dhaka_weather.csv`.
